# Introduction to Big Data Tools

**What you'll learn:** What "big data" means in practice, why pandas has limits, and how three tools — PySpark, Dask, and DuckDB — solve different parts of the problem.

**Prerequisites:** [01 - Python Basics](01-python-basics.ipynb)

This notebook is mostly reading and context. The goal is to help you choose the right tool before diving into the hands-on notebooks that follow.

## What Is "Big Data"?

There's no exact size that makes data "big." In practice, data becomes big when it:

- **Doesn't fit in memory** — your computer has a fixed amount of RAM (e.g. 8 GB or 16 GB). If your data file is 20 GB, pandas can't load it all at once.
- **Takes too long to process** — even if data fits in memory, a single-threaded tool might take hours to run a query on billions of rows.
- **Keeps growing** — new data arrives constantly (sensor readings, web traffic, financial transactions), so the processing pipeline must keep up.

At an internship, you might deal with datasets ranging from a few hundred megabytes to hundreds of gigabytes (or more). The tools in this series help you handle all of those.

## Why pandas isn't enough

pandas is fantastic for data exploration and analysis — it's what you learned in Notebook 01. But it has two fundamental limits:

1. **Everything lives in RAM.** When you call `pd.read_csv()`, pandas loads the entire file into your computer's memory. A 5 GB CSV file might use 10-15 GB of RAM once parsed.

2. **Single-threaded.** pandas uses one CPU core at a time. Your MacBook has 8+ cores, but pandas only uses one of them.

Let's see a quick demo with a small file, then think about what happens when files get large:

In [ ]:
import pandas as pd
import time

# Time how long it takes to read our small sales file
start = time.time()
df = pd.read_csv("../data/sales.csv")
elapsed = time.time() - start

print(f"Rows: {len(df):,}")
print(f"Time to load: {elapsed:.4f} seconds")
print(f"Memory used: {df.memory_usage(deep=True).sum() / 1024:.1f} KB")

500 rows is trivial. Now imagine:

| Rows | Approximate CSV size | pandas feasible? |
|------|---------------------|-------------------|
| 500 | ~30 KB | Instant |
| 100,000 | ~6 MB | Fast |
| 10,000,000 | ~600 MB | Slow, high RAM |
| 100,000,000 | ~6 GB | Might crash |
| 1,000,000,000 | ~60 GB | Won't fit in RAM |

The tools we'll learn next are designed for exactly this situation.

## The Three Tools

Each tool takes a different approach to the same problem: how to analyze data that's too large or too slow for pandas.

| | **PySpark** | **Dask** | **DuckDB** |
|---|---|---|---|
| **What it is** | Python API for Apache Spark, a distributed computing engine | A parallel computing library that extends pandas | An in-process analytical database (like SQLite, but for analytics) |
| **Best for** | Very large data on clusters; industry standard in big data | Scaling pandas workflows across multiple cores | Fast SQL queries on local files, quick analysis |
| **Approach** | Distributed across many machines | Parallel across cores on one machine | Optimized single-machine SQL engine |
| **Interface** | Its own DataFrame API + SQL | pandas-like API | SQL (with a Python wrapper) |
| **Learning curve** | Moderate — new API to learn | Low — very similar to pandas | Low — if you know SQL |
| **Scales to** | Petabytes across clusters | Multi-core, moderate data | Single machine, tens of GB |
| **When to choose** | Your internship uses Spark; data lives on a cluster | You know pandas and want to scale up without changing your code much | You want fast SQL queries on CSV/Parquet files without setting up a server |

## PySpark — The Industry Standard

**Apache Spark** is the most widely used big data processing engine. It was designed to run across a **cluster** (many computers working together), but it also runs locally on your laptop for learning and prototyping.

**PySpark** is simply the Python API for Spark. Under the hood, Spark is written in Scala/Java and runs on the JVM (Java Virtual Machine), which is why our environment includes Java.

**When you'll use it:** many companies (especially in finance, tech, and consulting) use Spark for their data pipelines. If your internship involves large-scale data processing, there's a good chance you'll encounter Spark.

**Trade-off:** more to learn upfront, but the most transferable skill in big data.

## Dask — pandas, but Parallel

**Dask** was created by people who love pandas but needed it to handle bigger data. It mimics the pandas API almost exactly, but splits your data into **partitions** and processes them in parallel.

The key idea in Dask is **lazy evaluation**: when you write `ddf.groupby("category").sum()`, Dask doesn't actually compute anything yet. It builds a plan of what to do. The computation only runs when you call `.compute()`. This lets Dask optimize the entire chain of operations before executing.

**When you'll use it:** when you already know pandas and your data is too big for it but doesn't require a full Spark cluster. It's great for prototyping and medium-scale work.

**Trade-off:** easiest transition from pandas, but doesn't scale as far as Spark.

## DuckDB — SQL Speed, Zero Setup

**DuckDB** is a newer tool that's become very popular. It's an **analytical database** that runs entirely inside your Python process — no server to install, no cluster to manage. You give it a SQL query and a CSV (or Parquet) file, and it runs the query extremely fast.

Think of it as an evolution of SQLite, but optimized for **analytical queries** (scanning many rows, computing aggregations) rather than transactional work (inserting one row at a time).

**When you'll use it:** for quick, ad-hoc analysis of local files. It's especially handy when you want to run SQL queries without first loading data into pandas.

**Trade-off:** single-machine only (can't distribute across a cluster), but impressively fast for what it does.

## A Quick Visual Summary

```
Data Size:   Small ──────────────── Medium ──────────────── Massive
             (MBs)                  (GBs)                   (TBs+)

pandas:      ████████████░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░
DuckDB:      ██████████████████████████████░░░░░░░░░░░░░░░░░
Dask:        ░░░░░░████████████████████████████████░░░░░░░░░
PySpark:     ░░░░░░░░░░████████████████████████████████████████
```

There's significant overlap. For many real-world tasks, more than one tool would work fine. The "right" choice often depends on what your team already uses and what skills you want to build.

## What's Next

The next three notebooks are hands-on with each tool. They all use the same dataset and perform similar operations so you can compare directly:

- [03 - Big Data with PySpark](03-big-data-pyspark.ipynb)
- [04 - Big Data with Dask](04-big-data-dask.ipynb)
- [05 - Big Data with DuckDB](05-big-data-duckdb.ipynb)

You can work through them in order, or jump to the one that interests you most. Each one is self-contained.

**Before continuing**, make sure you've generated the large dataset by running this command in your terminal:

```bash
python scripts/generate_large_data.py
```

This creates a 100,000-row file at `data/sales_large.csv` that the big data notebooks use.